# 「맥(脈)」 — 소비데이터 기반 지방소멸 조기경보 분석

제1회 AI금융빅데이터플랫폼 소비데이터 활용 분석·아이디어 공모전

이 노트북은 BC카드 제공 소비데이터로 **전처리 → 지표 산출 → 유형 판정**까지를 수행하고,
제안서에 인용된 모든 수치를 재현한다.

## 분석 설계 요약

| 축 | 지표 | 근거 |
|---|---|---|
| 축 1 | 상권기능 사다리 (0~3) | 상권은 취약한 업종부터 순서대로 소실된다 |
| 축 2 | 세대이탈지수 (%p) | 근린소매 연령 분포는 실존 인구 구조를 반영한다 |

두 축의 조합으로 255개 시군구를 **2×2 유형**으로 진단한다.

## 목차
1. 환경 설정과 데이터 적재
2. 데이터 구조 진단
3. 전처리
4. 업종 결측 구조 — 사다리 설계의 근거
5. 축 1 · 상권기능 사다리
6. 축 2 · 세대이탈지수
7. 지표 검증
8. 2×2 유형 판정
9. 외국인 소비 코드의 정체 규명
10. 시각화
11. 폐기된 지표의 기록
12. 산출물 저장

## 1. 환경 설정과 데이터 적재

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# 데이터 경로 — 환경에 맞게 수정
CANDIDATES = [
    Path("ABP_CONTEST_DATA.csv"),
    Path("data/ABP_CONTEST_DATA.csv"),
    Path("/mnt/user-data/uploads/ABP_CONTEST_DATA.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, f"데이터 파일을 찾을 수 없습니다. 후보: {CANDIDATES}"

OUT = Path("output"); OUT.mkdir(exist_ok=True)
print("데이터:", DATA_PATH.resolve())

In [ ]:
raw = pd.read_csv(DATA_PATH, dtype={"GENDER_CD": str, "AGE_CD": str})
print(f"{raw.shape[0]:,}행 × {raw.shape[1]}열")
raw.head(3)

## 2. 데이터 구조 진단

분석 설계 전에 데이터가 실제로 무엇을 담고 있는지 확인한다.
특히 **소멸 분석의 주 대상인 군 지역이 누락되지 않았는지**가 핵심이다.

In [ ]:
print("── 결측 ──")
print(f"전체 결측값: {raw.isna().sum().sum()}개")

print("\n── 코드 체계 ──")
for c in ["STRD_YYMM", "GENDER_CD", "AGE_CD", "TP_BUZ_NO"]:
    v = sorted(map(str, raw[c].unique()))
    print(f"{c:12s} n={len(v):2d}  {v}")

print("\n── 비공개 처리 임계 ──")
print(f"cnt 최솟값 {raw.cnt.min()}건 · amt 최솟값 {raw.amt.min():,}원")
print("→ 건수 11건 미만 셀은 미수록된 것으로 보인다 (k-익명화)")

In [ ]:
raw["region"] = raw.SIDO_NM + " " + raw.CCG_NM
cover = raw.groupby("SIDO_NM").CCG_NM.nunique().sort_values(ascending=False)
print(f"총 시군구: {raw.region.nunique()}개\n")
print(cover.to_string())

In [ ]:
# 소멸 분석의 대상인 '군' 지역 포함 여부
guns = sorted({r for r in raw.region.unique() if r.split()[-1].endswith("군")})
print(f"군 지역 {len(guns)}개 포함")
print("예시:", ", ".join(guns[:12]))

In [ ]:
# (지역 × 업종) 조합별 관측 월 수 — 시계열 완전성
dom_raw = raw[raw.GENDER_CD.isin(["1", "2"])]
pairs = dom_raw.groupby(["region", "TP_BUZ_NM"]).STRD_YYMM.nunique()
n_months = raw.STRD_YYMM.nunique()
print(pairs.value_counts().sort_index().to_string())
print(f"\n6개월 완전 관측: {(pairs == n_months).sum():,}/{len(pairs):,} "
      f"({(pairs == n_months).mean():.1%})")

**진단 결과**

- 결측값 0개, 255개 시군구 전수, 군 지역 누락 없음
- (지역×업종) 조합의 97.6%가 6개월 완전 관측
- 건수 11건 미만 셀은 비공개 처리 → 소규모 지역의 희소 업종이 **행 단위로 사라진다**

마지막 항목이 중요하다. 결측이 무작위가 아니라 **상권 규모와 연동**되므로,
결측 자체가 정보를 담고 있을 가능성이 있다. 4절에서 확인한다.

## 3. 전처리

In [ ]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # 업종명 공백 제거: '편 의 점' → '편의점'
    out["buz"] = out.TP_BUZ_NM.str.replace(" ", "", regex=False)
    out["region"] = out.SIDO_NM + " " + out.CCG_NM
    out["GENDER_CD"] = out.GENDER_CD.astype(str).str.lower()
    out["AGE_CD"] = out.AGE_CD.astype(str).str.lower()
    out["STRD_YYMM"] = out.STRD_YYMM.astype(int)
    # 세그먼트 플래그
    out["is_domestic"] = out.GENDER_CD.isin(["1", "2"])
    out["is_foreign"] = out.GENDER_CD == "3"
    out["is_unknown"] = (out.GENDER_CD == "x") | (out.AGE_CD == "x")
    return out


df = preprocess(raw)
dom = df[df.is_domestic]          # 내국인 — 본 분석의 기본 모집단
print(f"업종명: {sorted(df.buz.unique())}")
print(f"\n내국인 {df.is_domestic.mean():.1%} · 외국인 {df.is_foreign.mean():.1%} "
      f"· 미상 {df.is_unknown.mean():.1%} (행 기준)")

In [ ]:
# 세그먼트별 금액 비중
seg = pd.DataFrame({
    "금액": [df.loc[df.is_domestic, "amt"].sum(),
             df.loc[df.is_foreign, "amt"].sum(),
             df.loc[df.is_unknown, "amt"].sum()],
}, index=["내국인", "외국인", "미상"])
seg["비중"] = (seg.금액 / df.amt.sum() * 100).round(2)
seg

미상 코드(`x`)는 성별과 연령이 **항상 동시에** 결측이다. 별도 세그먼트로 분리하되
연령 기반 지표에서는 제외한다. 외국인은 7.4%로 무시할 수 없는 규모여서 9절에서 따로 다룬다.

## 4. 업종 결측 구조 — 사다리 설계의 근거

11개 업종이 지역별로 존재하는지 전수 확인한다.
결측이 무작위라면 지역 특성과 무관해야 한다.

In [ ]:
piv = dom.pivot_table(index="region", columns="buz", values="amt", aggfunc="sum")
missing = piv.isna().sum().sort_values(ascending=False)
tbl = pd.DataFrame({"결측 지역 수": missing,
                    "결측률": (missing / len(piv) * 100).round(1)})
tbl

결측은 명백히 무작위가 아니다. 편의점·슈퍼마켓·일반한식 등 7개 업종은 **전 지역 존재**하는
반면, 한정식은 85%, 갈비전문점은 59%가 결측이다.

가설: 상권 기능이 축소될 때 **취약한 업종부터 순서대로** 사라진다면,
업종 존재 패턴에 포함 관계(위계)가 나타나야 한다.

In [ ]:
pres = piv.notna()


def containment(inner: str, outer: str) -> float:
    """inner가 있는 지역 중 outer도 있는 비율."""
    return pres.loc[pres[inner], outer].mean() * 100


checks = [("대형할인점", "일식회집"), ("갈비전문점", "대형할인점"),
          ("한정식", "갈비전문점"), ("한정식", "대형할인점")]
for a, b in checks:
    print(f"{a} 보유 지역 중 {b} 보유 비율: {containment(a, b):5.1f}%")

**일식회집 ← 대형할인점 ← 갈비전문점** 순의 포함 관계가 확인된다.
대형할인점이 있으면 일식회집이 100% 존재하고, 갈비전문점이 있으면 대형할인점이 93.3% 존재한다.

반면 **한정식은 위계에서 벗어난다.** 한정식 보유 지역 중 갈비전문점 보유율이 53.8%에
그친다. 전주·안동 등 향토음식 도시의 **지역 고유 특성**으로 판단해 사다리에서 제외한다.

## 5. 축 1 · 상권기능 사다리

In [ ]:
LADDER = ["일식회집", "대형할인점", "갈비전문점"]   # 하위 → 상위 (뒤쪽이 먼저 소실)

ladder_bits = pres[LADDER].astype(int)
metrics = pd.DataFrame(index=piv.index)
metrics["ladder_score"] = ladder_bits.sum(axis=1)
metrics["ladder_pattern"] = ladder_bits.astype(str).agg("".join, axis=1)
metrics["missing_tiers"] = ladder_bits.apply(
    lambda r: ", ".join([c for c in LADDER if r[c] == 0]) or "-", axis=1)

# 정합 조합 = 하위부터 순서대로 채워진 형태
CONFORM = {"".join(["1"] * k + ["0"] * (len(LADDER) - k)) for k in range(len(LADDER) + 1)}
metrics["ladder_conforms"] = metrics.ladder_pattern.isin(CONFORM)

print(metrics.ladder_score.value_counts().sort_index().to_string())
print(f"\n위계 정합: {metrics.ladder_conforms.sum()}/{len(metrics)} "
      f"({metrics.ladder_conforms.mean():.1%})")

In [ ]:
combo = metrics.ladder_pattern.value_counts()
names = {"000": "없음", "100": "일식회집", "110": "+대형할인점", "111": "+갈비전문점"}
pd.DataFrame({
    "조합": combo.index,
    "구성": [names.get(c, "위계 이탈") for c in combo.index],
    "시군구 수": combo.values,
    "정합": [c in CONFORM for c in combo.index],
}).sort_values(["정합", "시군구 수"], ascending=[False, False]).reset_index(drop=True)

255개 중 248개(97.3%)가 위계에 부합한다. 이탈한 7개 지역은 일식회집과 갈비전문점은
있으나 대형할인점만 없는 형태로, 인접 대도시에 광역 유통이 흡수된 경우로 해석된다.

## 6. 축 2 · 세대이탈지수

핵심 가정: **편의점과 슈퍼마켓은 원정 소비가 없다.** 아무도 우유를 사러 옆 시군구로
가지 않는다. 따라서 근린소매의 연령 분포는 그 지역에 **실제로 존재한 사람의 연령 구조**를
반영한다.

이를 기준선으로 놓고 외식 소비의 연령 분포와 비교하면, 차이가 나는 세대가
지역 경제에서 빠져나간 세대다.

In [ ]:
DINING = ["갈비전문점", "한정식", "일식회집", "일반한식",
          "중국음식", "스넥", "제과점", "서양음식"]
RETAIL = ["편의점", "슈퍼마켓"]
AGE_LABEL = {"1": "20대 이하", "2": "20대", "3": "30대",
             "4": "40대", "5": "50대", "6": "60대 이상"}

d_age = dom[dom.AGE_CD != "x"]


def age_share(types):
    sub = d_age[d_age.buz.isin(types)]
    t = sub.groupby(["region", "AGE_CD"]).amt.sum().unstack().fillna(0.0)
    return t.div(t.sum(axis=1), axis=0)


dining_share = age_share(DINING)
retail_share = age_share(RETAIL)
gap = (dining_share - retail_share) * 100      # %p

print(f"지역×연령 셀 결측: 외식 {dining_share.isna().sum().sum()}개 / "
      f"소매 {retail_share.isna().sum().sum()}개 (총 {gap.size}셀)")

In [ ]:
# 세대이탈지수 = 고령 잔존 − 청년 이탈
metrics["drain_index"] = gap["6"] - (gap["2"] + gap["3"])
for c in gap.columns:
    metrics[f"gap_{AGE_LABEL.get(c, c)}"] = gap[c]

metrics.drain_index.describe().round(2)

## 7. 지표 검증

지표가 의도한 것을 측정하는지 확인한다. 양극단이 상식과 맞아야 한다.

In [ ]:
top = metrics.drain_index.sort_values(ascending=False).head(8)
bot = metrics.drain_index.sort_values().head(8)
pd.DataFrame({
    "청년 이탈·고령 잔존 (상위)": top.index, "지수↑": top.values.round(1),
    "청년 유입 (하위)": bot.index, "지수↓": bot.values.round(1),
})

상위는 옹진군·신안군·곡성군 등 전형적 소멸 위험 군 지역, 하위는 서울 중구·용산구·
마포구 등 청년 상권 집중 지역이다. 양극단이 모두 예상과 일치한다.

In [ ]:
# 계절 강건성 — 6개월 데이터로 구조 진단이 가능한가
from scipy.stats import spearmanr

monthly = {}
for m in sorted(dom.STRD_YYMM.unique()):
    sub = dom[dom.STRD_YYMM == m].pivot_table(
        index="region", columns="buz", values="amt", aggfunc="sum").fillna(0)
    monthly[m] = (sub[DINING].sum(axis=1) / sub[RETAIL].sum(axis=1))

M = pd.DataFrame(monthly).dropna()
rho = M.corr(method="spearman")
print(f"1월 vs 6월 순위상관: {spearmanr(M.iloc[:, 0], M.iloc[:, -1]).statistic:.3f}")
print(f"전체 월 쌍 최소 상관: {rho.values[np.triu_indices(len(rho), 1)].min():.3f}")

idx = dom.groupby("STRD_YYMM").amt.sum()
print("\n월별 전국 소비지수 (평균=1)")
print((idx / idx.mean()).round(3).to_string())

구조 지표의 월간 순위상관이 0.95 이상으로 유지된다. 2월(설 연휴)과 5월(가정의달)의
변동이 ±10% 수준이어서 **6개월 데이터로도 구조 진단이 가능**하다는 근거가 된다.

## 8. 2×2 유형 판정

In [ ]:
DRAIN_THRESHOLD = 8.0      # 세대이탈지수 경보선 (%p)
LADDER_DEFICIT = 1         # 이 값 이하면 상권 기능 결손

deficit = metrics.ladder_score <= LADDER_DEFICIT
drained = metrics.drain_index > DRAIN_THRESHOLD

metrics["region_type"] = np.select(
    [deficit & drained, deficit & ~drained, ~deficit & drained],
    ["복합 소멸형", "기능 결손형", "이탈 선행형"],
    default="유지형")

TYPE_ORDER = ["유지형", "이탈 선행형", "기능 결손형", "복합 소멸형"]
metrics.region_type.value_counts().reindex(TYPE_ORDER).to_frame("시군구 수")

In [ ]:
# 두 축이 서로를 대체하는가 — 선형 단계 모형의 타당성 검토
r = spearmanr(metrics.ladder_score, metrics.drain_index).statistic
print(f"사다리 점수 vs 세대이탈지수 순위상관: {r:.3f}")
print("→ 약한 음의 관계이나 서로를 대체하지 못한다. 선형 단계가 아닌 2×2 유형이 타당하다.")

In [ ]:
for t in TYPE_ORDER[1:]:
    names = (metrics[metrics.region_type == t]
             .sort_values("drain_index", ascending=False).index[:8])
    print(f"[{t}] {', '.join(names)}\n")

**이탈 선행형 21개 지역이 이 서비스의 존재 이유다.** 상권 기능은 아직 남아 있어
기존 통계로는 정상으로 분류되지만, 소비 구조상 이미 세대 이탈이 진행 중인 곳이다.

## 9. 외국인 소비 코드의 정체 규명

`GENDER_CD = 3`은 공고에 정의가 없다. 두 가지 해석이 가능하다.

- **해석 A** 국내 발급 외국인 명의 카드 → 등록 체류 외국인 (정주)
- **해석 B** 해외 발급 카드 매입 건 → 관광객

지역 분포와 연령 구조로 판별한다. 관광객이라면 명동이 있는 서울 중구가 압도적이어야 한다.

In [ ]:
f = df[df.is_foreign].groupby("region").amt.sum()
total = df.groupby("region").amt.sum()
fshare = (f / total * 100).sort_values(ascending=False)

print("── 외국인 소비 비중 상위 12 ──")
print(fshare.head(12).round(1).to_string())
print(f"\n서울특별시 중구: {fshare.get('서울특별시 중구', float('nan')):.1f}% "
      f"(순위 {list(fshare.index).index('서울특별시 중구') + 1})")
print(f"서울특별시 강남구: {fshare.get('서울특별시 강남구', float('nan')):.1f}%")

In [ ]:
fa = df[df.is_foreign & (df.AGE_CD != "x")].groupby("AGE_CD").amt.sum()
da = df[df.is_domestic & (df.AGE_CD != "x")].groupby("AGE_CD").amt.sum()
cmp = pd.DataFrame({"외국인 %": (fa / fa.sum() * 100).round(1),
                    "내국인 %": (da / da.sum() * 100).round(1)})
cmp.index = [AGE_LABEL.get(i, i) for i in cmp.index]
print(cmp.to_string())

mf = df[df.is_foreign].groupby("STRD_YYMM").amt.sum()
print("\n월별 외국인 소비지수 (평균=1)")
print((mf / mf.mean()).round(3).to_string())

**판정: 해석 A (등록 체류 외국인)**

- 상위 지역이 영암(대불산단)·음성·진천·화성 만세구·안산 단원구 등 **산업단지 배후**에 집중
- **서울 중구가 상위권 밖, 강남구 4.1%** → 해외 발급 카드 가설 기각
- 20~30대 비중이 내국인의 1.8배 → 근로 연령 구조
- 월별 변동이 평탄 → 관광 성수기 신호 없음

In [ ]:
# 20~30대 소비를 누가 담당하는가
y = df[df.AGE_CD.isin(["2", "3"])]
youth_share = (y[y.is_foreign].groupby("region").amt.sum()
               / y.groupby("region").amt.sum() * 100).dropna().sort_values(ascending=False)

metrics["foreign_share"] = fshare.reindex(metrics.index).fillna(0)
metrics["youth_foreign_share"] = youth_share.reindex(metrics.index).fillna(0)

print(f"전국 중앙값: {youth_share.median():.1f}%\n")
print(youth_share.head(10).round(1).to_string())

In [ ]:
r = spearmanr(metrics.drain_index, metrics.foreign_share).statistic
print(f"세대이탈지수 vs 외국인 비중 순위상관: {r:.3f}")
print("→ 상관이 약하다. '외국인 정주가 소멸을 완충한다'고 일반화할 수 없다.")
print("   주장 범위는 '특정 산업 기반 지역에서 청년 소비 담당자가 교체되고 있다'까지로 한정한다.")

## 10. 시각화

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# 한글 폰트 — 환경에 있는 것을 자동 탐색
for cand in ["NotoSansCJK", "NanumGothic", "Malgun", "AppleGothic"]:
    hits = [p for p in fm.findSystemFonts() if cand in p]
    if hits:
        fm.fontManager.addfont(hits[0])
        plt.rcParams["font.family"] = fm.FontProperties(fname=hits[0]).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

NAVY, GREY, RED, AMBER = "#1F3A5F", "#9AA5B1", "#C0392B", "#D68910"
COLOR = {"유지형": GREY, "이탈 선행형": RED, "기능 결손형": AMBER, "복합 소멸형": NAVY}

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.6))
rng = np.random.default_rng(7)
for t, c in COLOR.items():
    s = metrics[metrics.region_type == t]
    ax.scatter(s.ladder_score + rng.normal(0, .11, len(s)), s.drain_index,
               s=26, c=c, alpha=.8, edgecolors="white", linewidths=.5,
               label=f"{t} ({len(s)})", zorder=3)
ax.axhline(DRAIN_THRESHOLD, color=NAVY, ls="--", lw=1.1)
ax.axvline(LADDER_DEFICIT + .5, color=NAVY, ls="--", lw=1.1)
ax.axhspan(DRAIN_THRESHOLD, 40, xmin=.5, color=RED, alpha=.05)
ax.set_xlabel("상권기능 사다리 점수"); ax.set_ylabel("세대이탈지수 (%p)")
ax.set_title("전국 255개 시군구의 2×2 유형 분포", color=NAVY, fontsize=13, pad=12)
ax.set_xticks([0, 1, 2, 3]); ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=.22); [ax.spines[s].set_visible(False) for s in ("top", "right")]
plt.tight_layout(); plt.savefig(OUT / "fig_type_matrix.png", dpi=200); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
for t, c in COLOR.items():
    idx = metrics[metrics.region_type == t].index
    prof = gap.loc[gap.index.intersection(idx)].mean()
    ax.plot(range(len(prof)), prof.values, marker="o", ms=5.5, lw=2.1, color=c, label=t)
ax.axhline(0, color="#555555", lw=1)
ax.set_xticks(range(len(gap.columns)))
ax.set_xticklabels([AGE_LABEL.get(c, c) for c in gap.columns])
ax.set_ylabel("세대 갭 (외식 − 근린소매, %p)")
ax.set_title("유형별 세대 이탈 프로파일", color=NAVY, fontsize=13, pad=11)
ax.legend(ncol=4, loc="upper center", fontsize=9)
ax.grid(alpha=.22); [ax.spines[s].set_visible(False) for s in ("top", "right")]
plt.tight_layout(); plt.savefig(OUT / "fig_generation_profile.png", dpi=200); plt.show()

In [ ]:
top10 = youth_share.head(10)[::-1]
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(range(len(top10)), top10.values, color=NAVY, height=.64)
ax.axvline(youth_share.median(), color=RED, ls="--", lw=1.4)
ax.text(youth_share.median() + 1.2, -.8, f"전국 중앙값 {youth_share.median():.1f}%",
        color=RED, fontsize=9)
for i, v in enumerate(top10.values):
    ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels([r.split(" ", 1)[-1] for r in top10.index])
ax.set_xlabel("20~30대 소비액 중 외국인 비중 (%)"); ax.set_xlim(0, 78)
ax.set_title("소멸 위험 지역의 청년 소비를 누가 떠받치는가", color=NAVY, fontsize=13, pad=11)
ax.grid(axis="x", alpha=.22); [ax.spines[s].set_visible(False) for s in ("top", "right")]
plt.tight_layout(); plt.savefig(OUT / "fig_youth_foreign.png", dpi=200); plt.show()

## 11. 폐기된 지표의 기록

초기 설계에는 있었으나 실측 검증을 통과하지 못해 폐기한 지표를 기록한다.
제안서 8장의 근거이며, 남은 지표가 왜 신뢰할 만한지를 보이기 위함이다.

In [ ]:
# 폐기 1 — 흡인 배율: (목적성 소비) ÷ (근린 소비)
attract = (piv[["갈비전문점", "한정식", "일식회집", "대형할인점"]].sum(axis=1)
           / piv[RETAIL].sum(axis=1))
check = ["부산광역시 북구", "경기도 여주시", "서울특별시 중구", "서울특별시 강남구"]
pd.DataFrame({
    "흡인배율": attract[check].round(3),
    "대형할인점(억)": (piv.loc[check, "대형할인점"] / 1e8).round(0),
    "근린소매(억)": (piv.loc[check, RETAIL].sum(axis=1) / 1e8).round(0),
    "전국순위": [int(attract.rank(ascending=False)[c]) for c in check],
})

**폐기 사유** — 대형할인점 유무가 값을 지배한다. 부산 북구가 대형할인점 770억 대
근린소매 209억으로 전국 1위인 반면, 대표적 흡인 지역인 서울 중구는 하위권으로 내려간다.
유동인구가 많아 편의점 소비(분모)가 커지는데 정작 대형할인점이 없기 때문이다.
측정 의도와 반대로 작동하므로 폐기한다.

In [ ]:
# 폐기 2 — 외식의존도: 도시 상권과 농촌 소멸지가 동시에 상위로 올라온다
dep = (piv[DINING].sum(axis=1) / piv[RETAIL].sum(axis=1)).sort_values(ascending=False)
pd.DataFrame({"외식의존도 상위 10": dep.head(10).index,
              "값": dep.head(10).values.round(2)})

**강등 사유** — 서울 종로·서초와 전남 함평·곡성이 동시에 상위에 위치한다.
농촌에서는 식료품이 현금·직거래로 빠져 분모가 축소되는 교란이 발생한다.
인구밀도나 고령화율로 층화하지 않으면 쓸 수 없으므로 보조 지표로 강등한다.

## 12. 산출물 저장

In [ ]:
cols = ["ladder_score", "ladder_pattern", "ladder_conforms", "missing_tiers",
        "drain_index", "region_type", "foreign_share", "youth_foreign_share"]
gap_cols = [c for c in metrics.columns if c.startswith("gap_")]
result = metrics[cols + gap_cols].copy()
result.index.name = "region"
result.to_csv(OUT / "region_diagnosis.csv", encoding="utf-8-sig")

summary = {
    "regions": int(len(result)),
    "ladder_conformity": round(float(result.ladder_conforms.mean()), 4),
    "type_distribution": result.region_type.value_counts().reindex(TYPE_ORDER).to_dict(),
    "drain_threshold": DRAIN_THRESHOLD,
    "youth_foreign_median": round(float(youth_share.median()), 1),
}
import json
(OUT / "notebook_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
result.head()

## 13. 엔진 결과와 대조 (선택)

`maek_engine.py`가 같은 디렉터리에 있으면 노트북 산출과 엔진 산출이 일치하는지 확인한다.
서비스 배포본과 분석본이 갈라지지 않도록 하는 회귀 검사다.

In [ ]:
try:
    import maek_engine as M

    pipe = M.MaekPipeline(M.EngineConfig(llm_enabled=False))
    eng = pipe.run(card_path=str(DATA_PATH))["classified"]

    joined = result[["ladder_score", "drain_index", "region_type"]].join(
        eng[["ladder_score", "drain_index", "region_type"]],
        lsuffix="_nb", rsuffix="_eng", how="inner")

    same_type = (joined.region_type_nb == joined.region_type_eng).mean()
    max_diff = (joined.drain_index_nb - joined.drain_index_eng).abs().max()
    print(f"유형 일치율   {same_type:.1%}")
    print(f"세대이탈 최대 오차 {max_diff:.2e}")
    assert same_type == 1.0 and max_diff < 1e-9, "노트북과 엔진 결과가 불일치합니다"
    print("\n✓ 노트북과 엔진의 산출이 완전히 일치합니다.")
except ImportError:
    print("maek_engine.py를 찾을 수 없어 대조를 건너뜁니다.")